# Crédito agregado y tasas — diagnóstico con fuentes observadas

Fuentes canónicas: SFC y BanRep en Master_DATOS. Este notebook no contiene ni recrea raw. **No constituye estimación de PD ni relación causal.**

Variables: $g_{t}^{YoY}=100(C_t/C_{t-4}-1)$, $LDR_t=C_t/(CC_t+AH_t+CDT_t)$ y $r_{t-1}$ como IBR observada tres meses antes del trimestre. Verificar series originales y disponibilidad histórica antes de backtesting.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Notebook CI checks structure even when canonical Drive files are unavailable.
# Never fabricate replacement observations or claim empirical results in this mode.
candidates = [Path.cwd(), *Path.cwd().parents]
root = next((p for p in candidates if (p/'data-governance/observed-financial-marts/src/financial_mart.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Repository root with financial_mart.py not found')
sys.path.insert(0, str(root/'data-governance/observed-financial-marts/src'))
from financial_mart import execute
sfc = os.getenv('SFC_CANONICAL_CSV')
rates = os.getenv('BANREP_CANONICAL_CSV')
if sfc and rates:
    out = os.getenv('OBSERVED_MART_OUTPUT', '/tmp/observed-financial-mart')
    manifest = execute(sfc, rates, out)
    print({k: manifest[k] for k in ('sfc_rows', 'banrep_months', 'mart_rows', 'paired_yoy_ibr_rows', 'max_quarter')})
    df = pd.read_csv(Path(out)/'observed_macro_credit_mart.csv', parse_dates=['fecha_corte'])
    assert df['source_class'].eq('OBSERVED_AGGREGATE_DERIVED').all()
    assert not df['is_obligor_pd'].astype(str).str.lower().eq('true').any()
else:
    print('NO_DATA: canonical Drive inputs not supplied. Empirical analysis skipped, no synthetic substitution.')
    df = pd.DataFrame(columns=['fecha_corte','crecimiento_cartera_yoy_pct','cartera_sobre_depositos','ibr_rezagado_pct'])


In [ ]:
if df.empty:
    print('SKIPPED: plots require verified observed data.')
else:
    fig, ax = plt.subplots(figsize=(10,4))
    ax.plot(df.fecha_corte, df.crecimiento_cartera_yoy_pct, label='Cartera nominal YoY (%)')
    ax.axhline(0, color='gray', linewidth=.7)
    ax.set(title='Cartera agregada SFC — crecimiento interanual', ylabel='% nominal')
    ax.legend()
    plt.show()
    fig, ax = plt.subplots(figsize=(10,4))
    ax.plot(df.fecha_corte, df.cartera_sobre_depositos, label='Cartera / depósitos seleccionados')
    ax.set(title='Proxy descriptivo de apalancamiento del sistema', ylabel='Ratio')
    ax.legend()
    plt.show()


In [ ]:
if df.empty:
    print('SKIPPED: descriptive association needs observed datasets.')
else:
    paired = df[['crecimiento_cartera_yoy_pct','ibr_rezagado_pct']].dropna()
    if len(paired) < 4:
        print('INSUFFICIENT_DATA: cannot compute Pearson correlation.')
    else:
        r = paired.corr().iloc[0,1]
        print('Pares', len(paired), 'Pearson:', round(r,6))
        fig, ax = plt.subplots(figsize=(6,4))
        ax.scatter(paired.ibr_rezagado_pct, paired.crecimiento_cartera_yoy_pct, alpha=.6)
        ax.set(xlabel='IBR rezagado (% anual)', ylabel='Crecimiento nominal cartera YoY (%)',
               title='Asociación descriptiva; no causal')
        plt.show()


## Interpretación y falsación

1. No se puede afirmar causalidad: simultaneidad de política monetaria, inflación, demanda, composición y cambios de régimen.
2. La razón cartera/depósitos usa cuenta corriente, ahorro y CDT: **no equivale a indicador prudencial de liquidez**.
3. Comparar resultados con transformaciones estacionarias, residuos HAC, tendencias y especificaciones con más controles antes de formular hipótesis predictivas.
4. Incorporar vintages de publicación, unidades y reconciliación SFC antes de usar la señal en simulaciones históricas.
5. Registrar resultados negativos y evitar interpolaciones sintéticas sobre fechas faltantes.